# Independent Verification of Bankai (Saravanan, 2026)

**Source under review:** [github.com/nikshepsvn/bankai](https://github.com/nikshepsvn/bankai), `experiments/06_generalization_search.py`

**Requires:** GPU runtime (T4 sufficient, ~8GB VRAM). Install: `pip install gguf transformers torch numpy matplotlib`

Bankai reports 4/17 held-out probe fixes with 0 breaks using row-level XOR flips on PrismML's 1-bit Bonsai 8B.

This notebook independently verifies those claims by:
1. **Scoring all 90 probes** on the GGUF model via a pure-PyTorch Q1_0 inference engine
2. **Calibrating each probe**: checking whether Bankai's hardcoded "wrong" tokens match the model's actual top wrong predictions
3. **Measuring integral probe degeneracy**: whether 15 probes testing the same token contrast constitute independent measurements
4. **Reproducing the search**: running Bankai's row-level greedy search and evaluating on held-out validation

I reproduced the experiment using a standalone PyTorch Q1_0 inference engine (no MLX dependency) and found two methodological issues worth noting.

82% of probes measure the wrong thing. Bankai hardcodes "wrong" tokens for each probe (e.g., " 0" for all polynomial derivatives). When I checked what the model actually predicts, 74/90 hardcoded wrong tokens don't match the model's top wrong prediction. The median rank of the hardcoded wrong token is 27th. The model has 26 other tokens it prefers more. This means most logit gap measurements reflect distance to an irrelevant token, not distance to the decision boundary.

The integral category is one measurement, not fifteen. All 15 integral probes (10 train + 5 validation) test the token contrast " 1" vs " 2". Improving the model's preference for " 1" over " 2" on any integral prompt trivially improves all others. The claimed cross-category generalization includes "integral" as an independent category, but it isn't.

The notebook is self-contained (runs on a free Colab T4) and reproduces Bankai's probes verbatim from the source. The baseline matches Bankai's claimed 17/30 wrong on validation, confirming cross-runtime consistency between GGUF/PyTorch and MLX.
The idea that 1-bit weights are sensitive to XOR perturbation is interesting, these notes are about the evaluation methodology, not that idea.



In [ ]:
!pip install -q gguf transformers torch numpy matplotlib huggingface_hub 2>/dev/null
import os, math, torch, numpy as np
import torch.nn.functional as F

## Minimal Q1_0 Inference Engine

Self-contained PyTorch implementation for loading and running PrismML's Bonsai GGUF models.
Handles Q1_0 dequantization (128-weight groups, 1-bit weights + FP16 per-group scales),
RoPE attention, and RMS normalization.

In [ ]:
GROUP_SIZE = 128

def patch_gguf_q1_0():
    import gguf
    from gguf.constants import GGMLQuantizationType, GGML_QUANT_SIZES
    try:
        GGMLQuantizationType(41)
    except ValueError:
        obj = int.__new__(GGMLQuantizationType, 41)
        obj._name_ = 'Q1_0'; obj._value_ = 41
        GGMLQuantizationType._member_map_['Q1_0'] = obj
        GGMLQuantizationType._value2member_map_[41] = obj
    if GGMLQuantizationType(41) not in GGML_QUANT_SIZES:
        GGML_QUANT_SIZES[GGMLQuantizationType(41)] = (128, 18)

def load_model(model_path, device):
    patch_gguf_q1_0()
    from gguf import GGUFReader
    reader = GGUFReader(model_path)
    meta = {}
    for field in reader.fields:
        try:
            parts = reader.fields[field].parts
            val = parts[-1].tolist() if hasattr(parts[-1], 'tolist') else parts[-1]
            if isinstance(val, list) and len(val) == 1: val = val[0]
            meta[field] = val
        except: pass
    tmap = {t.name: t for t in reader.tensors}
    def _m(*keys, default):
        for k in keys:
            if k in meta: return meta[k]
        return default
    cfg = {
        'n_layers':   _m('llama.block_count','qwen2.block_count','qwen3.block_count', default=36),
        'n_heads':    _m('llama.attention.head_count','qwen2.attention.head_count','qwen3.attention.head_count', default=32),
        'n_kv_heads': _m('llama.attention.head_count_kv','qwen2.attention.head_count_kv','qwen3.attention.head_count_kv', default=8),
        'hidden_dim': _m('llama.embedding_length','qwen2.embedding_length','qwen3.embedding_length', default=4096),
        'rms_eps': 1e-6,
        'rope_theta': _m('llama.rope.freq_base','qwen2.rope.freq_base','qwen3.rope.freq_base', default=1000000.0),
    }
    hd = _m('llama.attention.key_length','qwen2.attention.key_length','qwen3.attention.key_length', default=None)
    cfg['head_dim'] = hd if hd else (cfg['hidden_dim'] // cfg['n_heads'])
    gate = [n for n in tmap if 'ffn_gate' in n and 'weight' in n][0]
    gs = tuple(int(s) for s in tmap[gate].shape)
    cfg['intermediate_dim'] = gs[1] if gs[0] == cfg['hidden_dim'] else gs[0]
    weights = {}
    for name, tensor in tmap.items():
        weights[name] = {
            'raw': torch.tensor(np.array(tensor.data)),
            'shape': tuple(int(s) for s in tensor.shape),
            'type': str(tensor.tensor_type).rsplit('.', 1)[-1],
        }
    print(f"Loaded: {cfg['n_layers']}L, {cfg['hidden_dim']}D, {len(weights)} tensors")
    return cfg, weights

In [ ]:
class Q1Engine:
    EMBED='token_embd.weight'; NORM='output_norm.weight'; LM_HEAD='output.weight'
    ATTN_Q='attn_q.weight'; ATTN_K='attn_k.weight'; ATTN_V='attn_v.weight'; ATTN_O='attn_output.weight'
    FFN_GATE='ffn_gate.weight'; FFN_UP='ffn_up.weight'; FFN_DOWN='ffn_down.weight'
    ATTN_NORM='attn_norm.weight'; FFN_NORM='ffn_norm.weight'

    def __init__(self, cfg, weights, device, search_layers=None):
        self.cfg, self.weights, self.device = cfg, weights, device
        self.search_layers = search_layers or [1,2,3,4,34]
        self._shifts = torch.arange(32, device=device, dtype=torch.int32)
        self._dequant_cache, self._unpacked_cache = {}, {}
        self.tokenizer = None
        vram = torch.cuda.get_device_properties(device).total_memory/1e9 if device.type=='cuda' else 0
        nL, hD, iD = cfg['n_layers'], cfg['hidden_dim'], cfg['intermediate_dim']
        nH, nKV, hdim = cfg['n_heads'], cfg['n_kv_heads'], cfg['head_dim']
        est_gb = (151936*hD + nL*(hD*nH*hdim + hD*nKV*hdim*2 + nH*hdim*hD + hD*iD*3))*2/1e9
        self.cache_dequant = (vram - est_gb) > 5.0
        print(f'VRAM {vram:.0f}GB, dequant ~{est_gb:.1f}GB, cache={"ON" if self.cache_dequant else "OFF"}')

    @staticmethod
    def ln(layer, suffix): return f'blk.{layer}.{suffix}'

    def unpack_q1_0(self, raw, out_f, in_f):
        rb = raw.to(torch.uint8).numpy().ravel()
        ng = (int(out_f)*int(in_f))//GROUP_SIZE
        bpg = GROUP_SIZE//8 + 2
        off = np.arange(ng)*bpg
        sr = np.zeros((ng,2), dtype=np.uint8)
        sr[:,0] = rb[off]; sr[:,1] = rb[off+1]
        scales = sr.view(np.float16).reshape(ng)
        br = np.zeros((ng,16), dtype=np.uint8)
        for i in range(16): br[:,i] = rb[off+2+i]
        packed = br.view(np.uint32).reshape(ng,4).view(np.int32)
        return (
            torch.tensor(packed, dtype=torch.int32, device=self.device),
            torch.tensor(scales, dtype=torch.float16, device=self.device),
        )

    def dequantize(self, packed, scales, out_f, in_f):
        ng = packed.shape[0]
        if int(out_f)*int(in_f) < 60_000_000:
            bits = ((packed.unsqueeze(-1) >> self._shifts) & 1).float().reshape(ng, GROUP_SIZE)
            return (scales.float().unsqueeze(1) * (2.*bits - 1.)).reshape(int(out_f), int(in_f))
        chunks = []
        for s in range(0, ng, 50000):
            e = min(s+50000, ng)
            b = ((packed[s:e].unsqueeze(-1) >> self._shifts) & 1).float().reshape(e-s, GROUP_SIZE)
            chunks.append((scales[s:e].float().unsqueeze(1) * (2.*b - 1.)).reshape(-1))
            del b
        return torch.cat(chunks).reshape(int(out_f), int(in_f))

    def load_fp(self, name):
        w = self.weights[name]
        raw = w['raw'].numpy().view(np.uint8).tobytes()
        dt = np.float16 if w['type'] in ('1','F16','f16') else np.float32
        return torch.tensor(np.frombuffer(raw, dtype=dt).reshape(w['shape']),
                            dtype=torch.float32, device=self.device)

    def get_weight(self, name):
        if name in self._dequant_cache: return self._dequant_cache[name]
        if name not in self.weights: return None
        w = self.weights[name]
        if w['type'] in ('Q1_0','q1_0','41'):
            out_f = w['shape'][1] if len(w['shape']) > 1 else 1
            in_f = w['shape'][0]
            if name not in self._unpacked_cache:
                p, s = self.unpack_q1_0(w['raw'], out_f, in_f)
                self._unpacked_cache[name] = (p, s, out_f, in_f)
            p, s, of, inf = self._unpacked_cache[name]
            r = self.dequantize(p, s, of, inf)
            if self.cache_dequant: self._dequant_cache[name] = r
            return r
        r = self.load_fp(name)
        self._dequant_cache[name] = r
        return r

    def linear(self, x, name):
        w = self.get_weight(name)
        r = x @ w.t()
        if not self.cache_dequant: del w; torch.cuda.empty_cache()
        return r

    @staticmethod
    def rms_norm(x, w, eps=1e-6):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + eps) * w

    def build_rope(self, seq_len):
        hd, th = self.cfg['head_dim'], self.cfg['rope_theta']
        pos = torch.arange(seq_len, device=self.device, dtype=torch.float32)
        freqs = 1.0 / (th ** (torch.arange(0, hd, 2, device=self.device, dtype=torch.float32) / hd))
        a = pos.unsqueeze(1) * freqs.unsqueeze(0)
        e = torch.cat([a, a], dim=-1)
        return e.cos(), e.sin()

    @staticmethod
    def apply_rope(x, cos, sin):
        s = x.shape[2]
        c = cos[:s].unsqueeze(0).unsqueeze(0)
        sn = sin[:s].unsqueeze(0).unsqueeze(0)
        d2 = x.shape[-1] // 2
        xr = torch.cat([-x[..., d2:], x[..., :d2]], dim=-1)
        return x * c + xr * sn

    @staticmethod
    def repeat_kv(x, n):
        if n == 1: return x
        b, nkv, s, hd = x.shape
        return x.unsqueeze(2).expand(b, nkv, n, s, hd).reshape(b, nkv*n, s, hd)

    @torch.no_grad()
    def forward(self, input_ids):
        c = self.cfg; sl = input_ids.shape[1]
        rc, rs = self.build_rope(sl)
        nr = c['n_heads'] // c['n_kv_heads']
        h = self.get_weight(self.EMBED)[input_ids[0]].unsqueeze(0).float()
        for L in range(c['n_layers']):
            n = self.rms_norm(h, self.get_weight(self.ln(L, self.ATTN_NORM)), c['rms_eps'])
            q = self.linear(n, self.ln(L, self.ATTN_Q)).view(1, sl, c['n_heads'], c['head_dim']).transpose(1, 2)
            k = self.linear(n, self.ln(L, self.ATTN_K)).view(1, sl, c['n_kv_heads'], c['head_dim']).transpose(1, 2)
            v = self.linear(n, self.ln(L, self.ATTN_V)).view(1, sl, c['n_kv_heads'], c['head_dim']).transpose(1, 2)
            q = self.rms_norm(q, self.get_weight(self.ln(L, 'attn_q_norm.weight')), c['rms_eps'])
            k = self.rms_norm(k, self.get_weight(self.ln(L, 'attn_k_norm.weight')), c['rms_eps'])
            q = self.apply_rope(q, rc, rs); k = self.apply_rope(k, rc, rs)
            k = self.repeat_kv(k, nr); v = self.repeat_kv(v, nr)
            a = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(c['head_dim'])
            if sl > 1:
                mask = torch.triu(torch.ones(sl, sl, device=self.device), 1).bool()
                a = a.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf'))
            a = F.softmax(a, dim=-1)
            ao = torch.matmul(a, v).transpose(1, 2).reshape(1, sl, c['hidden_dim'])
            h = h + self.linear(ao, self.ln(L, self.ATTN_O))
            nf = self.rms_norm(h, self.get_weight(self.ln(L, self.FFN_NORM)), c['rms_eps'])
            g = self.linear(nf, self.ln(L, self.FFN_GATE))
            u = self.linear(nf, self.ln(L, self.FFN_UP))
            h = h + self.linear(F.silu(g) * u, self.ln(L, self.FFN_DOWN))
            del q, k, v, a, ao, g, u, n, nf
            if not self.cache_dequant: torch.cuda.empty_cache()
        h = self.rms_norm(h, self.get_weight(self.NORM), c['rms_eps'])
        lm = self.get_weight(self.LM_HEAD)
        if lm is None: lm = self.get_weight(self.EMBED)
        return h @ lm.t()

    def load_tokenizer(self):
        from transformers import AutoTokenizer
        self.tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-8B', trust_remote_code=True)

    def flip_group(self, layer, proj, group_idx):
        name = self.ln(layer, proj)
        if name not in self._unpacked_cache:
            w = self.weights[name]
            of, inf = w['shape'][1], w['shape'][0]
            p, s = self.unpack_q1_0(w['raw'], of, inf)
            self._unpacked_cache[name] = (p, s, of, inf)
        self._unpacked_cache[name][0][group_idx] ^= -1
        if name in self._dequant_cache: del self._dequant_cache[name]

## Load Bonsai 8B

In [ ]:
from huggingface_hub import hf_hub_download
model_path = './Bonsai-8B.gguf'
if not os.path.exists(model_path):
    hf_hub_download(repo_id='prism-ml/Bonsai-8B-gguf', filename='Bonsai-8B.gguf', local_dir='.')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
cfg, weights = load_model(model_path, device)
engine = Q1Engine(cfg, weights, device, search_layers=[1,2,3,4,34])
engine.load_tokenizer()

## Bankai's Probes: Verbatim from `experiments/06_generalization_search.py`

60 training + 30 validation probes across 6 categories. Reproduced exactly as specified.

In [ ]:
def mp(name, prompt, correct, wrong, category):
    return {'name':name, 'prompt':prompt, 'correct':correct, 'wrong':wrong, 'category':category}

TRAIN = [
    mp('pd_train_0','d/dx [x^4 + 3x^2] =',' 4',' 0','poly_deriv'),
    mp('pd_train_1','d/dx [x^5 + 2x^3] =',' 5',' 0','poly_deriv'),
    mp('pd_train_2','d/dx [x^3 + 7x] =',' 3',' 0','poly_deriv'),
    mp('pd_train_3','d/dx [2x^4] =',' 8',' 0','poly_deriv'),
    mp('pd_train_4','d/dx [x^6] =',' 6',' 0','poly_deriv'),
    mp('pd_train_5','d/dx [x^2 + x] =',' 2',' 0','poly_deriv'),
    mp('pd_train_6','d/dx [3x^3 + x^2] =',' 9',' 0','poly_deriv'),
    mp('pd_train_7','d/dx [x^3 + 2x^2 + x] =',' 3',' 0','poly_deriv'),
    mp('pd_train_8','The derivative of x^4 + 2x^3 is',' 4',' 0','poly_deriv'),
    mp('pd_train_9','Differentiate x^5 + x with respect to x:',' 5',' 0','poly_deriv'),
    mp('sd_train_0','The second derivative of x^4 is 12x^','2','3','second_deriv'),
    mp('sd_train_1','The second derivative of x^5 is',' 20',' 0','second_deriv'),
    mp('sd_train_2','The second derivative of x^3 is',' 6',' 0','second_deriv'),
    mp('sd_train_3','d^2/dx^2 [x^4] =',' 12',' 0','second_deriv'),
    mp('sd_train_4','d^2/dx^2 [x^5] =',' 20',' 0','second_deriv'),
    mp('sd_train_5','d^2/dx^2 [x^3] =',' 6',' 0','second_deriv'),
    mp('sd_train_6',"f(x) = x^4, f''(x) ="," 12"," 0","second_deriv"),
    mp('sd_train_7','The second derivative of x^6 is',' 30',' 0','second_deriv'),
    mp('sd_train_8','The second derivative of 2x^4 is',' 24',' 0','second_deriv'),
    mp('sd_train_9','Find the second derivative of x^5:',' 20',' 0','second_deriv'),
    mp('int_train_0','The integral of x^2 dx = ',' 1',' 2','integral'),
    mp('int_train_1','The integral of x^3 dx =',' 1',' 2','integral'),
    mp('int_train_2','The integral of x^4 dx =',' 1',' 2','integral'),
    mp('int_train_3','The integral of x dx =',' 1',' 2','integral'),
    mp('int_train_4','The integral of x^5 dx =',' 1',' 2','integral'),
    mp('int_train_5','Evaluate the integral of x^2 dx:',' 1',' 2','integral'),
    mp('int_train_6','Find the antiderivative of x^2:',' 1',' 2','integral'),
    mp('int_train_7','The antiderivative of x^3 is',' 1',' 2','integral'),
    mp('int_train_8','Integrate x^4 dx:',' 1',' 2','integral'),
    mp('int_train_9','The indefinite integral of x^3 dx =',' 1',' 2','integral'),
    mp('pr_train_0','Is 97 prime? Answer:',' Yes',' No','prime'),
    mp('pr_train_1','Is 101 prime? Answer:',' Yes',' No','prime'),
    mp('pr_train_2','Is 89 prime? Answer:',' Yes',' No','prime'),
    mp('pr_train_3','Is 83 prime? Answer:',' Yes',' No','prime'),
    mp('pr_train_4','Is 71 prime? Answer:',' Yes',' No','prime'),
    mp('pr_train_5','Is 91 prime? Answer:',' No',' Yes','prime'),
    mp('pr_train_6','Is 87 prime? Answer:',' No',' Yes','prime'),
    mp('pr_train_7','Is 95 prime? Answer:',' No',' Yes','prime'),
    mp('pr_train_8','Is 67 prime? Answer:',' Yes',' No','prime'),
    mp('pr_train_9','Is 99 prime? Answer:',' No',' Yes','prime'),
    mp('trig_train_0','sin(pi/6) =',' 1',' 0','trig'),
    mp('trig_train_1','sin(pi/2) =',' 1',' 0','trig'),
    mp('trig_train_2','cos(0) =',' 1',' 0','trig'),
    mp('trig_train_3','sin(0) =',' 0',' 1','trig'),
    mp('trig_train_4','cos(pi) =',' -',' 0','trig'),
    mp('trig_train_5','tan(pi/4) =',' 1',' 0','trig'),
    mp('trig_train_6','sin(pi) =',' 0',' 1','trig'),
    mp('trig_train_7','cos(pi/2) =',' 0',' 1','trig'),
    mp('trig_train_8','tan(0) =',' 0',' 1','trig'),
    mp('trig_train_9','cos(2*pi) =',' 1',' 0','trig'),
    mp('ed_train_0','d/dx [e^(2x)] = ',' 2',' 6','exp_deriv'),
    mp('ed_train_1','d/dx [e^(3x)] =',' 3',' 9','exp_deriv'),
    mp('ed_train_2','d/dx [e^(4x)] =',' 4',' 16','exp_deriv'),
    mp('ed_train_3','d/dx [e^x] =',' e',' x','exp_deriv'),
    mp('ed_train_4','d/dx [e^(-x)] =',' -',' e','exp_deriv'),
    mp('ed_train_5','The derivative of e^(2x) is',' 2',' 4','exp_deriv'),
    mp('ed_train_6','The derivative of e^(3x) is',' 3',' 9','exp_deriv'),
    mp('ed_train_7',"f(x) = e^(2x), f'(x) ="," 2"," 4","exp_deriv"),
    mp('ed_train_8','Differentiate e^(2x):',' 2',' 4','exp_deriv'),
    mp('ed_train_9','The derivative of e^x is',' e',' x','exp_deriv'),
]

VAL = [
    mp('pd_val_0','d/dx [x^7 + x] =',' 7',' 0','poly_deriv'),
    mp('pd_val_1','d/dx [4x^3] =',' 12',' 0','poly_deriv'),
    mp('pd_val_2','d/dx [x^4 + x^3 + x^2] =',' 4',' 0','poly_deriv'),
    mp('pd_val_3','Find the derivative of 2x^5 + x:',' 10',' 0','poly_deriv'),
    mp('pd_val_4',"f(x) = x^6 + 3x, f'(x) ="," 6"," 0","poly_deriv"),
    mp('sd_val_0','d^2/dx^2 [x^7] =',' 42',' 0','second_deriv'),
    mp('sd_val_1','The second derivative of x^3 + x^2 is',' 6',' 0','second_deriv'),
    mp('sd_val_2',"f(x) = x^5, f''(x) ="," 20"," 0","second_deriv"),
    mp('sd_val_3','The second derivative of 3x^3 is',' 18',' 0','second_deriv'),
    mp('sd_val_4',"What is f''(x) if f(x) = x^6?"," 30"," 0","second_deriv"),
    mp('int_val_0','The integral of x^6 dx =',' 1',' 2','integral'),
    mp('int_val_1','The integral of x^7 dx =',' 1',' 2','integral'),
    mp('int_val_2','What is the integral of x^2 dx?',' 1',' 2','integral'),
    mp('int_val_3','The antiderivative of x^4 is',' 1',' 2','integral'),
    mp('int_val_4','The integral of x^2 with respect to x is',' 1',' 2','integral'),
    mp('pr_val_0','Is 107 prime? Answer:',' Yes',' No','prime'),
    mp('pr_val_1','Is 113 prime? Answer:',' Yes',' No','prime'),
    mp('pr_val_2','Is 53 prime? Answer:',' Yes',' No','prime'),
    mp('pr_val_3','Is 77 prime? Answer:',' No',' Yes','prime'),
    mp('pr_val_4','Is 119 prime? Answer:',' No',' Yes','prime'),
    mp('trig_val_0','sin(pi/4) =',' 0',' 1','trig'),
    mp('trig_val_1','cos(pi/3) =',' 0',' 1','trig'),
    mp('trig_val_2','sin(pi/3) =',' 0',' 1','trig'),
    mp('trig_val_3','cos(pi/6) =',' 0',' 1','trig'),
    mp('trig_val_4','sin(3*pi/2) =',' -',' 0','trig'),
    mp('ed_val_0','d/dx [e^(5x)] =',' 5',' 25','exp_deriv'),
    mp('ed_val_1',"f(x) = e^(3x), f'(x) ="," 3"," 9","exp_deriv"),
    mp('ed_val_2','d/dx [e^(-2x)] =',' -',' 4','exp_deriv'),
    mp('ed_val_3','d/dx [2*e^x] =',' 2',' x','exp_deriv'),
    mp('ed_val_4','d/dx [e^(x^2)] =',' 2',' e','exp_deriv'),
]

CONTROLS = [
    mp('france','The capital of France is',' Paris',' London','knowledge'),
    mp('japan','The capital of Japan is',' Tokyo',' Beijing','knowledge'),
    mp('sky','The color of the sky is',' blue',' red','knowledge'),
    mp('einstein','Einstein is famous for the theory of',' relativity',' evolution','knowledge'),
    mp('water','The chemical formula for water is H','2','3','knowledge'),
]

print(f'{len(TRAIN)} training, {len(VAL)} validation, {len(CONTROLS)} control probes')

## Helper Functions

In [ ]:
def score_probe(engine, probe):
    tok = engine.tokenizer
    ids = tok.encode(probe['prompt'], add_special_tokens=False)
    cid = tok.encode(probe['correct'], add_special_tokens=False)[-1]
    wid = tok.encode(probe['wrong'], add_special_tokens=False)[-1]
    with torch.no_grad():
        logits = engine.forward(torch.tensor([ids], dtype=torch.long, device=engine.device))
    return logits[0, -1, cid].item() - logits[0, -1, wid].item()

def calibrate_probe(engine, probe):
    tok = engine.tokenizer
    ids = tok.encode(probe['prompt'], add_special_tokens=False)
    cid = tok.encode(probe['correct'], add_special_tokens=False)[-1]
    with torch.no_grad():
        logits = engine.forward(torch.tensor([ids], dtype=torch.long, device=engine.device))
        last = logits[0, -1, :]
    sorted_ids = last.argsort(descending=True)
    actual_wid = next(i.item() for i in sorted_ids[:10] if i.item() != cid)
    hc_wid = tok.encode(probe['wrong'], add_special_tokens=False)[-1]
    rank = (sorted_ids == hc_wid).nonzero()
    return {
        'name': probe['name'],
        'hardcoded_wrong': probe['wrong'], 'hardcoded_wrong_id': hc_wid,
        'hardcoded_wrong_rank': rank[0].item() if len(rank) > 0 else -1,
        'actual_wrong': tok.decode([actual_wid]), 'actual_wrong_id': actual_wid,
        'gap_hardcoded': last[cid].item() - last[hc_wid].item(),
        'gap_calibrated': last[cid].item() - last[actual_wid].item(),
    }

## Phase 1: Baseline Evaluation

Score all 90 probes. Bankai claims 17/30 wrong on validation.

In [ ]:
print('Training (60):')
train_gaps = {p['name']: score_probe(engine, p) for p in TRAIN}
wrong_t = sum(1 for g in train_gaps.values() if g <= 0)
print(f'  {wrong_t}/{len(train_gaps)} wrong')

print('\nValidation (30):')
val_gaps = {p['name']: score_probe(engine, p) for p in VAL}
wrong_v = sum(1 for g in val_gaps.values() if g <= 0)
print(f'  {wrong_v}/{len(val_gaps)} wrong')
print(f'  Bankai claims: 17/30')

print('\nPer-category (validation):')
from collections import defaultdict
cats = defaultdict(lambda: {'w': 0, 't': 0})
for p in VAL:
    cats[p['category']]['t'] += 1
    if val_gaps[p['name']] <= 0: cats[p['category']]['w'] += 1
for c in sorted(cats):
    print(f'  {c:<15} {cats[c]["w"]}/{cats[c]["t"]} wrong')

## Phase 2: Calibration Audit

For each probe, find the model's **actual** top wrong prediction and compare to Bankai's
hardcoded wrong token. If they don't match, the logit gap measures distance to an irrelevant token.

In [ ]:
all_cal = [calibrate_probe(engine, p) for p in TRAIN + VAL]
matches = sum(1 for c in all_cal if c['hardcoded_wrong_id'] == c['actual_wrong_id'])
print(f'Hardcoded == actual top wrong: {matches}/{len(all_cal)} ({100*matches/len(all_cal):.1f}%)')
print(f'Mismatches: {len(all_cal)-matches}/{len(all_cal)}\n')

print(f'{"Name":<20} {"Hardcoded":>10} {"Actual":>10} {"Rank":>6} {"Gap(HC)":>8} {"Gap(Cal)":>8}')
print('-' * 66)
for c in all_cal:
    flag = ' ' if c['hardcoded_wrong_id'] == c['actual_wrong_id'] else '*'
    print(f'{flag}{c["name"]:<19} {repr(c["hardcoded_wrong"]):>10} {repr(c["actual_wrong"]):>10} '
          f'{c["hardcoded_wrong_rank"]:>6} {c["gap_hardcoded"]:>+8.3f} {c["gap_calibrated"]:>+8.3f}')

## Phase 3: Visualizing the Calibration Gap

In [ ]:
import matplotlib.pyplot as plt

hc = [c['gap_hardcoded'] for c in all_cal]
cal = [c['gap_calibrated'] for c in all_cal]
ranks = [c['hardcoded_wrong_rank'] for c in all_cal if c['hardcoded_wrong_rank'] >= 0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = {'pd':'tab:blue', 'sd':'tab:orange', 'int':'tab:red',
          'pr':'tab:green', 'trig':'tab:purple', 'ed':'tab:brown'}
labels = {'pd':'poly_deriv', 'sd':'second_deriv', 'int':'integral',
          'pr':'prime', 'trig':'trig', 'ed':'exp_deriv'}
for pfx, col in colors.items():
    m = [c['name'].startswith(pfx) for c in all_cal]
    ax1.scatter([hc[i] for i, v in enumerate(m) if v],
               [cal[i] for i, v in enumerate(m) if v],
               c=col, label=labels[pfx], alpha=0.7, s=40)
lim = max(abs(min(hc+cal)), abs(max(hc+cal))) * 1.1
ax1.plot([-lim, lim], [-lim, lim], 'k--', alpha=0.3)
ax1.axhline(0, color='gray', lw=0.5); ax1.axvline(0, color='gray', lw=0.5)
ax1.set_xlabel('Hardcoded gap (Bankai)')
ax1.set_ylabel('Calibrated gap')
ax1.set_title('Logit gap: hardcoded vs calibrated')
ax1.legend(fontsize=8)

ax2.hist(ranks, bins=30, color='steelblue', edgecolor='white')
ax2.axvline(1, color='red', ls='--', label='Rank 1 (correct target)')
ax2.set_xlabel('Rank of hardcoded wrong token')
ax2.set_ylabel('Count')
ax2.set_title("Where Bankai's wrong tokens actually rank")
ax2.legend()
plt.tight_layout(); plt.show()

print(f'Median rank: {np.median(ranks):.0f}, Mean: {np.mean(ranks):.1f}, '
      f'At rank 1: {sum(r <= 1 for r in ranks)}/{len(ranks)}')

## Phase 4: Integral Probe Degeneracy

All 15 integral probes test `" 1"` vs `" 2"`. One token contrast = one effective measurement, not 15.

In [ ]:
int_probes = [p for p in TRAIN + VAL if p['category'] == 'integral']
int_gaps = [(p['name'], score_probe(engine, p)) for p in int_probes]

print('Integral probes: all test " 1" vs " 2":')
for n, g in int_gaps:
    print(f'  {n:<20} {g:>+8.4f}')

gaps = [g for _, g in int_gaps]
print(f'\nStd: {np.std(gaps):.4f}')
print(f'All same sign: {all(g > 0 for g in gaps) or all(g <= 0 for g in gaps)}')
print(f'\nVerdict: {len(int_probes)} probes, 1 unique token contrast = 1 effective measurement')

## Phase 5: Search Reproduction (optional)

Reproduces Bankai's Experiment 6: greedy row-level search, 300 iterations.
Set `RUN_SEARCH = True` to execute (~15-30 min on T4).

In [ ]:
PROJ_MAP = {
    'gate_proj': 'ffn_gate.weight',
    'up_proj': 'ffn_up.weight',
    'down_proj': 'ffn_down.weight',
}

def flip_row(engine, layer, proj, row_idx):
    proj_name = PROJ_MAP[proj]
    for g in range(row_idx * 32, row_idx * 32 + 32):
        engine.flip_group(layer, proj_name, g)

RUN_SEARCH = False  # Set True to run (~15-30 min)

if RUN_SEARCH:
    import random, time
    random.seed(42); np.random.seed(42)
    t_bl = {p['name']: score_probe(engine, p) for p in TRAIN}
    c_bl = {p['name']: score_probe(engine, p) for p in CONTROLS}
    print(f'Baseline: {sum(1 for g in t_bl.values() if g <= 0)}/{len(t_bl)} wrong')

    candidates, wts = [], []
    for layer in [1, 2, 3, 4, 34]:
        for proj in ['gate_proj', 'up_proj']:
            name = engine.ln(layer, PROJ_MAP[proj])
            if name not in engine._unpacked_cache:
                w = engine.weights[name]
                of, inf = w['shape'][1], w['shape'][0]
                p, s = engine.unpack_q1_0(w['raw'], of, inf)
                engine._unpacked_cache[name] = (p, s, of, inf)
            _, scales, _, _ = engine._unpacked_cache[name]
            n_rows = scales.shape[0] // 32
            rs = scales.float().abs().reshape(n_rows, 32).mean(1)
            for r in range(n_rows):
                candidates.append((layer, proj, r))
                wts.append(rs[r].item())
    wts = np.array(wts); wts /= wts.sum()
    print(f'Search space: {len(candidates)} rows')

    accepted, best_f, tried = [], 0.0, set()
    for i in range(300):
        a = 0
        while a < 100:
            idx = np.random.choice(len(candidates), p=wts)
            key = candidates[idx]
            if key not in tried: tried.add(key); break
            a += 1
        else: print('Exhausted'); break
        layer, proj, row = key
        flip_row(engine, layer, proj, row)
        tg = {p['name']: score_probe(engine, p) for p in TRAIN}
        cg = {p['name']: score_probe(engine, p) for p in CONTROLS}
        t_imp = sum(tg[n] - t_bl[n] for n in t_bl) / len(t_bl)
        c_deg = sum(max(0, c_bl[n] - cg[n]) for n in c_bl) / len(c_bl)
        f = t_imp - 2.0 * c_deg
        if f > best_f:
            accepted.append({'layer': layer, 'proj': proj, 'row': row})
            best_f = f
            fixes = sum(1 for n in t_bl if t_bl[n] <= 0 and tg[n] > 0)
            brk = sum(1 for n in t_bl if t_bl[n] > 0 and tg[n] <= 0)
            print(f'  [{i+1:>4}/300] ACCEPT L{layer}.{proj}[{row}] '
                  f'f={f:+.4f} flips={len(accepted)} fixes={fixes} breaks={brk}')
        else:
            flip_row(engine, layer, proj, row)

    # Evaluate validation
    val_now = {p['name']: score_probe(engine, p) for p in VAL}
    for fl in reversed(accepted):
        flip_row(engine, fl['layer'], fl['proj'], fl['row'])
    val_bl = {p['name']: score_probe(engine, p) for p in VAL}
    for fl in accepted:
        flip_row(engine, fl['layer'], fl['proj'], fl['row'])
    fixed = [n for n in val_bl if val_bl[n] <= 0 and val_now[n] > 0]
    broke = [n for n in val_bl if val_bl[n] > 0 and val_now[n] <= 0]
    print(f'\nValidation: fixed={len(fixed)}, broke={len(broke)}')
    print(f'Bankai claims: fixed=4, broke=0')
else:
    print('Search skipped. Set RUN_SEARCH = True.')

## Summary

| Issue | Finding |
|-------|---------|
| Probe calibration | ~82% of hardcoded wrong tokens don't match the model's actual top wrong prediction. Median rank: 27. |
| Integral degeneracy | All 15 integral probes test `" 1"` vs `" 2"`, 1 effective measurement, not 15 |
| Cross-runtime | GGUF/PyTorch baseline matches Bankai's claimed 17/30 wrong on validation |
| Search reproduction | Row-level flips produce 0 boundary crossings in 300 iterations |

The core observation, that 1-bit weights are sensitive to XOR perturbation, is valid.
The methodology for measuring and claiming generalization from that sensitivity has significant limitations.